# Ch9. ARIMA Models
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

The cells below follow the slide deck *Ch9. ARIMA models* (updated, 73 slides) and are meant to be run from top to bottom: later cells reuse objects created in earlier ones. Cells marked *figure code* reproduce a chart whose code is not shown on the slide.

In [ ]:
%pip install -q statsforecast utilsforecast statsmodels matplotlib

In [ ]:
BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"  # datasets in this repository (data/)

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.rcParams.update({"figure.figsize": (7, 3.5)})

## Data
Slides 11 and 35 use a monthly series `y` / `train` without showing how it is loaded. Here we use US employment in leisure and hospitality (textbook Section 9.9, also loaded again on slide 52).

In [ ]:
leisure = (
    pd.read_csv(BASE + "us_employment.csv", parse_dates=["ds"])
    .loc[lambda x:
         (x["unique_id"] == "Leisure and Hospitality")
         & (x["ds"] >= "2001")]
    .reset_index(drop=True)
    .assign(y=lambda x: x["y"] / 1e3)   # millions
    [["unique_id", "ds", "y"]]
)
y = leisure["y"].values
train = leisure

## [Slide 11] 9.1 Automated Differencing

In [ ]:
from statsforecast.arima import ndiffs, nsdiffs

# Number of first differences (uses sequence of KPSS tests)
d  = ndiffs(y, alpha=0.05, test="kpss", max_d=2)

# Number of seasonal differences
# (uses seasonal strength measure;
#  recommends D=1 if Fs >= 0.64)
D  = nsdiffs(y, period=12, test="seas", max_D=1)

print(f"d = {d}, D = {D}")

## [Slide 14] 9.2 AR Model Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import ArmaProcess

np.random.seed(1)
n = 100  # burnin=50: drop the first 50 draws

# AR(1): phi_1 = -0.8  => ArmaProcess AR poly: 1 + 0.8*B
ar1_proc = ArmaProcess([1, 0.8], [1])
y_ar1 = ar1_proc.generate_sample(nsample=n, burnin=50)

np.random.seed(2)
# AR(2): phi_1 = 1.3, phi_2 = -0.7
# => poly: 1 - 1.3*B + 0.7*B^2
ar2_proc = ArmaProcess([1, -1.3, 0.7], [1])
y_ar2 = ar2_proc.generate_sample(nsample=n, burnin=50)

## [Slide 15] 9.2 AR Model Simulation: Stationarity Check

In [ ]:
# True True: all roots outside the unit circle
print(ar1_proc.isstationary, ar2_proc.isstationary)
print(ar1_proc.arroots)          # [-1.25]
print(ar2_proc.arroots)          # 0.929 +/- 0.753j
print(np.abs(ar2_proc.arroots))  # [1.195 1.195] > 1

# A random walk (phi_1 = 1) is not stationary
print(ArmaProcess([1, -1], [1]).isstationary)  # False

## [Slide 16] 9.2 AR Model Simulation (cont.)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

series = [(y_ar1, "steelblue", r"AR(1): $\phi_1 = -0.8$"),
          (y_ar2, "darkorange",
           r"AR(2): $\phi_1=1.3,\ \phi_2=-0.7$")]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for j, (y, c, ttl) in enumerate(series):
    axes[0, j].plot(y, lw=1.2, color=c)
    axes[0, j].axhline(0, color="gray", lw=0.8, ls="--")
    axes[0, j].set(title=ttl, xlabel="$t$")
    plot_acf(y, lags=24, ax=axes[1, j], title="ACF",
             zero=False, bartlett_confint=False)
plt.tight_layout(); plt.show()

## [Slide 20] 9.3 MA Model Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import ArmaProcess

np.random.seed(3)
# MA(1): theta_1 = 0.8  => MA poly: 1 + 0.8*B
ma1_proc = ArmaProcess([1], [1, 0.8])
y_ma1 = ma1_proc.generate_sample(nsample=100, burnin=50)

np.random.seed(4)
# MA(2): theta_1 = -1, theta_2 = 0.8
# => poly: 1 - B + 0.8*B^2
ma2_proc = ArmaProcess([1], [1, -1, 0.8])
y_ma2 = ma2_proc.generate_sample(nsample=100, burnin=50)

## [Slide 21] 9.3 MA Model Simulation (cont.)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

series = [(y_ma1, "steelblue", r"MA(1): $\theta_1 = 0.8$"),
          (y_ma2, "darkorange",
           r"MA(2): $\theta_1=-1,\ \theta_2=0.8$")]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for j, (y, c, ttl) in enumerate(series):
    axes[0, j].plot(y, lw=1.2, color=c)
    axes[0, j].axhline(0, color="gray", lw=0.8, ls="--")
    axes[0, j].set(title=ttl, xlabel="$t$")
    plot_acf(y, lags=24, ax=axes[1, j], title="ACF",
             zero=False, bartlett_confint=False)
plt.tight_layout(); plt.show()

## [Slide 25] 9.4 ACF/PACF Identification — Code (1)

In [ ]:
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(5)
ar2 = ArmaProcess([1, -0.6, -0.3], [1])  # phi = 0.6, 0.3
y_ar = ar2.generate_sample(nsample=200, burnin=50)

np.random.seed(6)
ma2 = ArmaProcess([1], [1, 0.6, 0.3])  # theta = 0.6, 0.3
y_ma = ma2.generate_sample(nsample=200, burnin=50)

## [Slide 26] 9.4 ACF/PACF Identification — Code (2)

In [ ]:
series = [(y_ar, "AR(2)"), (y_ma, "MA(2)")]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.5))
for i, (y, name) in enumerate(series):
    plot_acf(y, lags=20, ax=axes[i, 0], zero=False,
             bartlett_confint=False, title=f"ACF: {name}")
    plot_pacf(y, lags=20, ax=axes[i, 1], zero=False,
              title=f"PACF: {name}")
plt.tight_layout(); plt.show()

## [Slide 35] 9.5 AutoARIMA Algorithm — Code

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import ARIMASummary

sf = StatsForecast(
    models=[AutoARIMA(season_length=12, stepwise=False)],
    freq="MS",
)
sf.fit(df=train)
arima_model = sf.fitted_[0, 0]
print(ARIMASummary(arima_model.model_))

## [Slide 38] 9.5 CAF Exports — Data

In [ ]:
import pandas as pd

caf_economy = (
    pd.read_csv(BASE + "global_economy.csv",
                parse_dates=["ds"])
    .rename(columns={"Exports": "y"})
    .loc[lambda x: x["Code"] == "CAF"]
    [["unique_id", "ds", "y"]]
)

## [Slide 39] 9.5 CAF Exports — Time Plot, ACF and PACF
*Figure code (not shown on the slide).*

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplot_mosaic([["y", "y"], ["acf", "pacf"]], figsize=(9, 4.6))
axes["y"].plot(caf_economy["ds"], caf_economy["y"], marker="o", ms=3, color="steelblue")
axes["y"].set(xlabel="Year", ylabel="% of GDP", title="Central African Republic exports")
plot_acf(caf_economy["y"], axes["acf"], zero=False, auto_ylims=True, bartlett_confint=False)
plot_pacf(caf_economy["y"], axes["pacf"], zero=False, auto_ylims=True)
axes["acf"].set(xlabel="lag [1Y]", ylabel="acf", title="ACF")
axes["pacf"].set(xlabel="lag [1Y]", ylabel="pacf", title="PACF")
for k in ("acf", "pacf"):
    axes[k].xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.show()

## [Slide 40] 9.5 CAF Exports — How Many Differences?

In [ ]:
from statsforecast.arima import ndiffs

ndiffs(caf_economy["y"].values)
# 1

## [Slide 41] 9.5 CAF Exports — Differenced Series (Fig. 9.14)
*Figure code (not shown on the slide).*

In [ ]:
df = caf_economy.assign(diff=caf_economy["y"].diff()).iloc[1:]
fig, axes = plt.subplot_mosaic([["diff", "diff"], ["acf", "pacf"]], figsize=(9, 4.6))
axes["diff"].plot(df["ds"], df["diff"], marker="o", ms=3, color="steelblue")
axes["diff"].set(xlabel="Year", ylabel="difference(Exports)")
plot_acf(df["diff"], axes["acf"], zero=False, auto_ylims=True, bartlett_confint=False)
plot_pacf(df["diff"], axes["pacf"], zero=False, auto_ylims=True)
axes["acf"].set(xlabel="lag [1Y]", ylabel="acf", title="ACF")
axes["pacf"].set(xlabel="lag [1Y]", ylabel="pacf", title="PACF")
for k in ("acf", "pacf"):
    axes[k].xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.show()

## [Slide 42] 9.5 CAF Exports — Four Candidate Models

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA
from statsforecast.arima import ARIMASummary

models = [ARIMA(order=(2, 1, 0), alias="arima210"),
          ARIMA(order=(0, 1, 3), alias="arima013"),
          AutoARIMA(alias="stepwise"),
          AutoARIMA(stepwise=False, alias="search")]
sf = StatsForecast(models=models, freq="YS")
sf.fit(caf_economy)

rows = []
for m in sf.fitted_[0]:   # one fitted model per column
    rows.append({"model": m,
                 "Orders": ARIMASummary(m.model_),
                 "sigma2": m.model_["sigma2"],
                 "loglik": m.model_["loglik"],
                 "aic": m.model_["aic"],
                 "aicc": m.model_["aicc"],
                 "bic": m.model_["bic"]})
pd.DataFrame(sorted(rows, key=lambda d: d["aicc"]))

## [Slide 44] 9.5 CAF Exports — Residuals (Fig. 9.15)
*Figure code (not shown on the slide).*

In [ ]:
fc = sf.forecast(df=caf_economy, h=10, fitted=True, level=[80, 95])
insample = sf.forecast_fitted_values()
residuals = insample["y"] - insample["search"]

fig, axes = plt.subplot_mosaic([["resid", "resid"], ["acf", "hist"]], figsize=(9, 4.6))
axes["resid"].plot(caf_economy["ds"], residuals, marker="o", ms=3, color="steelblue")
axes["resid"].set(xlabel="Year", ylabel="Innovation residuals")
plot_acf(residuals, axes["acf"], zero=False, auto_ylims=True, bartlett_confint=False)
axes["acf"].xaxis.set_major_locator(MaxNLocator(integer=True))
axes["acf"].set(xlabel="lag [1Y]", ylabel="acf", title="ACF")
axes["hist"].hist(residuals, bins=20, color="steelblue", edgecolor="white")
axes["hist"].set(xlabel="residuals", ylabel="count", title="Histogram")
plt.tight_layout(); plt.show()

## [Slide 45] 9.5 CAF Exports — Ljung-Box Test

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

fc = sf.forecast(df=caf_economy, h=10,
                 fitted=True, level=[80, 95])
insample = sf.forecast_fitted_values()
# residuals of "search" = ARIMA(3,1,0)
residuals = insample["y"] - insample["search"]
acorr_ljungbox(residuals, lags=[10], model_df=3)
#     lb_stat  lb_pvalue
# 10     5.70      0.575

## [Slide 46] 9.5 CAF Exports — Forecasts (Fig. 9.16)
*Figure code (not shown on the slide).*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(caf_economy["ds"], caf_economy["y"], color="black", lw=1.2, label="Exports")
ax.plot(fc["ds"], fc["search"], color="steelblue", lw=1.5, label="ARIMA(3,1,0) forecast")
for lv, a in ((95, 0.15), (80, 0.3)):
    ax.fill_between(fc["ds"], fc[f"search-lo-{lv}"], fc[f"search-hi-{lv}"],
                    color="steelblue", alpha=a, label=f"{lv}% PI")
ax.set(xlabel="Year", ylabel="% of GDP", title="Central African Republic exports")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

## [Slide 53] 9.6 Example: US Leisure Employment — Data

In [ ]:
leisure = (
    pd.read_csv(BASE + "us_employment.csv", parse_dates=["ds"])
    .loc[lambda x:
         (x["unique_id"] == "Leisure and Hospitality")
         & (x["ds"] >= "2001")]
    .reset_index(drop=True)
    .assign(y=lambda x: x["y"] / 1e3)   # millions
    [["unique_id", "ds", "y"]]
)

## [Slide 54] 9.6 Example: Leisure Employment (Fig. 9.18)
*Figure code (not shown on the slide).*

In [ ]:
from matplotlib.ticker import MaxNLocator

def ts_acf_pacf(ds, y, title, ylabel, lag_label="lag [1M]"):
    fig, axes = plt.subplot_mosaic([["y", "y"], ["acf", "pacf"]], figsize=(9, 4.6))
    axes["y"].plot(ds, y, marker="o", ms=2.5, color="steelblue")
    axes["y"].set(xlabel="Month", ylabel=ylabel, title=title)
    plot_acf(y, axes["acf"], lags=36, zero=False, auto_ylims=True, bartlett_confint=False)
    plot_pacf(y, axes["pacf"], lags=36, zero=False, auto_ylims=True)
    axes["acf"].set(xlabel=lag_label, ylabel="acf", title="ACF")
    axes["pacf"].set(xlabel=lag_label, ylabel="pacf", title="PACF")
    for k in ("acf", "pacf"):
        axes[k].xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.tight_layout(); plt.show()

ts_acf_pacf(leisure["ds"], leisure["y"],
            "US employment: leisure and hospitality", "Millions of people")

## [Slide 55] 9.6 Example: Seasonally Differenced (Fig. 9.19)
*Figure code (not shown on the slide).*

In [ ]:
sdiff = leisure.assign(d=leisure["y"].diff(12)).iloc[12:]
ts_acf_pacf(sdiff["ds"], sdiff["d"], "Seasonally differenced: $(1-B^{12})y_t$", "")

## [Slide 56] 9.6 Example: Doubly Differenced (Fig. 9.20)
*Figure code (not shown on the slide).*

In [ ]:
ddiff = leisure.assign(d=leisure["y"].diff(12).diff()).iloc[13:]
ts_acf_pacf(ddiff["ds"], ddiff["d"], "Doubly differenced: $(1-B)(1-B^{12})y_t$", "")

## [Slide 57] 9.6 Example: Leisure — How Many Differences?

In [ ]:
from statsforecast.arima import ndiffs, nsdiffs

y = leisure["y"].values
nsdiffs(y, period=12)          # seasonal differences
# 1
y_sdiff = y[12:] - y[:-12]     # (1 - B^12) y_t
ndiffs(y_sdiff)                # further first differences
# 1

## [Slide 58] 9.6 Example: Leisure — Four Candidate Models

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA
from statsforecast.arima import ARIMASummary

models = [
    ARIMA(order=(0, 1, 2), seasonal_order=(0, 1, 1),
          season_length=12, alias="arima012011"),
    ARIMA(order=(2, 1, 0), seasonal_order=(0, 1, 1),
          season_length=12, alias="arima210011"),
    ARIMA(order=(2, 1, 0), seasonal_order=(1, 1, 0),
          season_length=12, alias="arima210110"),
    AutoARIMA(stepwise=False, approximation=False,
              season_length=12, alias="auto"),
]
sf = StatsForecast(models=models, freq="MS")
sf.fit(df=leisure)

rows = []
for m in sf.fitted_[0]:   # one fitted model per column
    rows.append({"model": m,
                 "Orders": ARIMASummary(m.model_),
                 "sigma2": m.model_["sigma2"],
                 "loglik": m.model_["loglik"],
                 "aic": m.model_["aic"],
                 "aicc": m.model_["aicc"],
                 "bic": m.model_["bic"]})
pd.DataFrame(sorted(rows, key=lambda d: d["aicc"]))

## [Slide 60] 9.6 Example: Leisure — Best Model and Ljung-Box

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# column 3 = "auto": ARIMA(2,1,0)(0,1,2)[12]
best = sf.fitted_[0, 3].model_
se = np.sqrt(np.diag(best["var_coef"]))
pd.DataFrame({"coef": best["coef"].values(), "s.e.": se},
             index=best["coef"].keys())

fc = sf.forecast(df=leisure, h=36, fitted=True, level=[80, 95])
insample = sf.forecast_fitted_values()
residuals = insample["y"] - insample["auto"]

# K = p + q + P + Q = 2 + 0 + 0 + 2 = 4 estimated parameters
acorr_ljungbox(residuals, lags=[24], model_df=4)

## [Slide 62] 9.6 Example: Leisure — Residuals (Fig. 9.21)
*Figure code (not shown on the slide).*

In [ ]:
fig, axes = plt.subplot_mosaic([["resid", "resid"], ["acf", "hist"]], figsize=(9, 4.6))
axes["resid"].plot(leisure["ds"], residuals, marker="o", ms=2.5, color="steelblue")
axes["resid"].set(xlabel="Month", ylabel="Innovation residuals")
plot_acf(residuals, axes["acf"], lags=36, zero=False, auto_ylims=True, bartlett_confint=False)
axes["acf"].xaxis.set_major_locator(MaxNLocator(integer=True))
axes["acf"].set(xlabel="lag [1M]", ylabel="acf", title="ACF")
axes["hist"].hist(residuals, bins=20, color="steelblue", edgecolor="white")
axes["hist"].set(xlabel="residuals", ylabel="count", title="Histogram")
plt.tight_layout(); plt.show()

## [Slide 63] 9.6 Example: Leisure — Forecasts (Fig. 9.22)
*Figure code (not shown on the slide).*

In [ ]:
last_year = leisure.iloc[-12:]          # training data: last 12 months only
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(last_year["ds"], last_year["y"], marker="o", ms=3, color="black", lw=1.2,
        label="Data (last 12 months)")
ax.plot(fc["ds"], fc["auto"], color="steelblue", lw=1.5,
        label="ARIMA(2,1,0)(0,1,2)$_{12}$ forecast")
for lv, a in ((95, 0.15), (80, 0.3)):
    ax.fill_between(fc["ds"], fc[f"auto-lo-{lv}"], fc[f"auto-hi-{lv}"],
                    color="steelblue", alpha=a, label=f"{lv}% PI")
ax.set(xlabel="Month", ylabel="Millions of people",
       title="US employment: leisure and hospitality")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

## [Slide 67] 9.7 Comparison 1: Australian Population

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS
from utilsforecast.losses import rmse, mae

aus_economy = (
    pd.read_csv(BASE + "aus_economy.csv", parse_dates=["ds"])
    .assign(y=lambda x: x["y"] / 1e6)    # millions
)
models = [AutoARIMA(), AutoETS()]
sf = StatsForecast(models=models, freq="YE")
cv_forecasts = sf.cross_validation(h=1, n_windows=10, df=aus_economy)

model_names = ["AutoARIMA", "AutoETS"]
rm = rmse(cv_forecasts, models=model_names, cutoff_col=None)
ma = mae(cv_forecasts, models=model_names, cutoff_col=None)
metrics = pd.concat([rm.assign(metric="RMSE"), ma.assign(metric="MAE")])
metrics[["metric", *model_names]]

## [Slide 69] 9.7 Comparison 2: Australian Quarterly Cement

In [ ]:
aus_production = pd.read_csv(BASE + "aus_production.csv",
                             parse_dates=["ds"])
cement = (
    aus_production.loc[lambda x: x["ds"] >= "1988"]
    [["ds", "Cement"]]
    .rename(columns={"Cement": "y"})
    .assign(unique_id="Cement")
)
train = (cement.loc[lambda x: x["ds"] < "2008"]
         .reset_index(drop=True))

## [Slide 70] 9.7 Comparison 2: Cement — ARIMA

In [ ]:
from statsforecast.arima import ARIMASummary
from statsmodels.stats.diagnostic import acorr_ljungbox

arima = AutoARIMA(season_length=4)
arima_sf = StatsForecast(models=[arima], freq="QS").fit(train)
arima_sf.forecast(df=train, h=10, fitted=True)
arima_insample = arima_sf.forecast_fitted_values()
arima_residuals = arima_insample["y"] - arima_insample["AutoARIMA"]

model = arima_sf.fitted_[0, 0].model_
print(ARIMASummary(model))
print(pd.Series(model["coef"]).round(3))
acorr_ljungbox(arima_residuals, lags=[8], model_df=5)

## [Slide 71] 9.7 Comparison 2: Cement — ETS

In [ ]:
ets = AutoETS(season_length=4)
ets_sf = StatsForecast(models=[ets], freq="QS").fit(train)
ets_sf.forecast(df=train, h=10, fitted=True)
ets_insample = ets_sf.forecast_fitted_values()
ets_residuals = ets_insample["y"] - ets_insample["AutoETS"]

model = ets_sf.fitted_[0, 0].model_
print(model["method"])
# alpha, gamma, l[0], s[0], s[1], s[2]
params = model["fit"].x
print(params.round(3))
acorr_ljungbox(ets_residuals, lags=[8])

## [Slide 72] 9.7 Comparison 2: Cement — Evaluation

In [ ]:
models = [AutoARIMA(season_length=4),
          AutoETS(season_length=4)]
sf = StatsForecast(models=models, freq="QS")

# (a) training/test: forecast the 10 test quarters
fc = sf.forecast(h=10, df=train, fitted=True).merge(cement)
insample = sf.forecast_fitted_values()
# (b) CV within the training set: 10 windows, h = 4
cv = sf.cross_validation(df=train, h=4, n_windows=10)

dfs = {"Train": insample, "Test": fc, "CV": cv}
metrics = pd.concat([
    metric(df, models=model_names, cutoff_col=None)
    .assign(metric=metric.__name__.upper(), type=type)
    for type, df in dfs.items() for metric in [rmse, mae]
])[["type", "metric", *model_names]]

## [Slide 73] 9.7 Comparison 2: Cement — Results

In [ ]:
metrics